# 한국어 Reasoning 튜닝 - Gemma 4 E4B + KOREAson YiSang (QLoRA 최적화판 v4)

## v3 → v4 핵심 수정 (QLoRA 함정 제거)

### 1. `lora_dropout = 0` (기존 0.05)
**이유**: 4bit 양자화는 weight에 이미 quantization noise를 주입한 상태. 여기에 dropout까지 더하면:
- Activation에 이중 stochastic noise → gradient SNR 악화
- Unsloth의 최적화 커널이 dropout=0 기준으로 튜닝됨 → 속도 저하
- QLoRA 원 논문 (Dettmers 2023) 및 Unsloth 공식 권장: **dropout=0**
- 짧은 학습(1~2 epoch)에서 dropout은 수렴 방해 요인

과적합 방어는 **weight_decay + EarlyStopping + NEFTune** 조합으로 충분합니다.

### 2. `use_rslora = True` (기존 False)
r=32 같이 rank가 어느 정도 클 때 rank-stabilized LoRA는 `alpha/sqrt(r)` 스케일링을 써서 학습이 더 안정적입니다.
일반 LoRA는 `alpha/r` 스케일링이라 rank 커질수록 effective LR이 불안정해져요.

### 3. `optim = "paged_adamw_8bit"` (기존 `adamw_8bit`)
QLoRA 논문 원조 권장. **메모리 스파이크를 CPU RAM으로 페이징**해서 OOM 방어. 
adamw_8bit보다 미세하게 느리지만 안정성이 크게 올라감 → 낮은 VRAM 환경에서 특히 유리.

### 4. 과적합 방어 전략 명시
dropout을 뺐기 때문에 다른 축으로 방어해야 합니다:
- `weight_decay = 0.01` (유지)
- `EarlyStoppingCallback(patience=3)` (유지)  
- `NEFTUNE_ALPHA = 5` (유지) — embedding noise로 정규화 효과
- `load_best_model_at_end = True` (유지) — overfit 직전 체크포인트 복원

## STEP 1. 패키지 설치
설치 후 **런타임 → 세션 다시 시작** 한 번 눌러주세요.

In [ ]:
!pip install -q --upgrade unsloth
!pip install -q --upgrade transformers trl peft accelerate bitsandbytes datasets wandb

# CUDA 런타임 버전 확인 (Gemma 4는 13.2 피할 것)
import subprocess
try:
    nvcc = subprocess.check_output(["nvcc", "--version"]).decode()
    print(nvcc)
    if "13.2" in nvcc:
        print("⚠️ CUDA 13.2 감지! Gemma 4 GGUF에서 품질 저하 보고됨.")
except Exception:
    pass

## STEP 2. HuggingFace / W&B 로그인

In [ ]:
from huggingface_hub import login
import wandb
from getpass import getpass

HF_TOKEN = getpass("HuggingFace token을 입력하세요: ")
WANDB_KEY = getpass("W&B API key를 입력하세요 (없으면 엔터): ")

login(HF_TOKEN)

if WANDB_KEY.strip():
    wandb.login(key=WANDB_KEY)
    REPORT_TO = "wandb"
else:
    REPORT_TO = "none"
    print("W&B 없이 진행합니다.")

## STEP 3. 설정값

### QLoRA 하이퍼파라미터 철학
4bit 양자화 상태에서는 **노이즈를 추가하지 말고**, **효과적인 학습 신호를 유지**하는 게 중요합니다.
그래서 dropout=0, rsLoRA=True, paged optimizer 조합을 씁니다.

### GPU별 권장 설정
| GPU | MAX_SEQ | BATCH | GRAD_ACCUM | SAMPLES | LORA_R |
|-----|---------|-------|------------|---------|--------|
| A100 40GB | 4096 | 2 | 4 | 20000 | 32 |
| L4 24GB | 3072 | 2 | 4 | 10000 | 32 |
| T4 16GB | 1536 | 1 | 8 | 5000 | 16 |

In [ ]:
from unsloth import FastLanguageModel
import torch
import os
import random
import numpy as np

SEED = 42

# ===== 모델 =====
MODEL_NAME = "unsloth/gemma-4-E4B-it"

# ===== 시퀀스/배치 (A100 기준) =====
MAX_SEQ_LENGTH = 4096
PER_DEVICE_BATCH_SIZE = 2
GRAD_ACCUM = 4

# ===== 데이터 =====
TRAIN_SAMPLES = 20000
VAL_RATIO = 0.02
MIN_TOKEN_LEN = 64

# ===== 학습 =====
NUM_EPOCHS = 2
LEARNING_RATE = 3e-5
WARMUP_RATIO = 0.03
WEIGHT_DECAY = 0.01              # 과적합 방어 1: weight decay
MAX_GRAD_NORM = 1.0
NEFTUNE_ALPHA = 5                # 과적합 방어 2: embedding noise (정규화 효과)

# ===== LoRA (QLoRA 최적화) =====
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0                 # 🔑 4bit에서는 0이 정답
USE_RSLORA = True                # 🔑 r이 클 때 안정적 스케일링

# ===== Optimizer (QLoRA 권장) =====
OPTIMIZER = "paged_adamw_8bit"   # 🔑 메모리 스파이크 방어

# ===== Thinking mode =====
ENABLE_THINKING_TRAINING = False

# ===== 출력 =====
OUTPUT_DIR = "outputs_gemma4_v4"
RUN_NAME = "ko-reason-gemma4-e4b-qlora-v4"

# ===== 재현성 =====
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print("=" * 55)
print(f"Model:            {MODEL_NAME}")
print(f"Max seq length:   {MAX_SEQ_LENGTH}")
print(f"Train samples:    {TRAIN_SAMPLES}")
print(f"Effective batch:  {PER_DEVICE_BATCH_SIZE * GRAD_ACCUM}")
print(f"LoRA r/alpha:     {LORA_R}/{LORA_ALPHA}")
print(f"LoRA dropout:     {LORA_DROPOUT}  (4bit 양자화 → 0 고정)")
print(f"rsLoRA:           {USE_RSLORA}")
print(f"Optimizer:        {OPTIMIZER}")
print(f"Learning rate:    {LEARNING_RATE}")
print(f"Epochs:           {NUM_EPOCHS}")
print(f"NEFTune alpha:    {NEFTUNE_ALPHA}")
print(f"Weight decay:     {WEIGHT_DECAY}")
print(f"Thinking SFT:     {ENABLE_THINKING_TRAINING}")
print("=" * 55)

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## STEP 4. 모델 로드 (4bit 양자화)

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = None,
    load_in_4bit = True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    print(f"pad_token을 eos_token({tokenizer.eos_token})으로 설정")

print(f"vocab size: {len(tokenizer)}")
print(f"eos_token: {tokenizer.eos_token} (id: {tokenizer.eos_token_id})")
print(f"pad_token: {tokenizer.pad_token} (id: {tokenizer.pad_token_id})")

## STEP 5. LoRA 어댑터 부착 (QLoRA 최적화)

### 핵심 포인트
- **`lora_dropout = 0`**: 4bit 환경에서 dropout은 이중 노이즈 문제. Unsloth 최적화도 0 기준.
- **`use_rslora = True`**: alpha/sqrt(r) 스케일링으로 r=32에서 더 안정적.
- **`bias = "none"`**: bias 학습은 4bit에서 불안정 가능성 + 파라미터 낭비.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_R,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = LORA_ALPHA,
    lora_dropout = LORA_DROPOUT,      # 0
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = SEED,
    use_rslora = USE_RSLORA,          # True
    loftq_config = None,
)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable_params:,} ({100*trainable_params/total_params:.2f}%)")
print(f"LoRA dropout:     {LORA_DROPOUT} (4bit 호환)")
print(f"rsLoRA enabled:   {USE_RSLORA}")

## STEP 6. 데이터셋 로드

In [ ]:
from datasets import load_dataset

raw_ds = load_dataset("KOREAson/YiSang-HighQuality", split="train")
print(raw_ds)
print("\n--- column names ---")
print(raw_ds.column_names)
print("\n--- sample[0] ---")
print(raw_ds[0])

## STEP 7. 데이터 포맷팅 + 길이 필터링 + train/val 분리

In [ ]:
def format_chat_example(example):
    messages = example["messages"]
    
    if ENABLE_THINKING_TRAINING:
        has_system = any(m.get("role") == "system" for m in messages)
        if not has_system:
            messages = [{"role": "system", "content": "<|think|>"}] + list(messages)
    
    text = tokenizer.apply_chat_template(
        messages,
        tokenize = False,
        add_generation_prompt = False,
    )
    return {"text": text}

ds = raw_ds.shuffle(seed=SEED)
if TRAIN_SAMPLES is not None:
    ds = ds.select(range(min(TRAIN_SAMPLES, len(ds))))

ds = ds.map(format_chat_example, remove_columns=raw_ds.column_names)

def add_length(example):
    tokens = tokenizer(example["text"], add_special_tokens=False)["input_ids"]
    return {"n_tokens": len(tokens)}

ds = ds.map(add_length, num_proc=2)

lengths = np.array(ds["n_tokens"])
print("=== 토큰 길이 분포 ===")
print(f"  min: {lengths.min()}")
print(f"  p25: {np.percentile(lengths, 25):.0f}")
print(f"  p50: {np.percentile(lengths, 50):.0f}")
print(f"  p75: {np.percentile(lengths, 75):.0f}")
print(f"  p95: {np.percentile(lengths, 95):.0f}")
print(f"  p99: {np.percentile(lengths, 99):.0f}")
print(f"  max: {lengths.max()}")
print(f"  >{MAX_SEQ_LENGTH} 초과: {(lengths > MAX_SEQ_LENGTH).sum()} / {len(lengths)}")
print(f"  <{MIN_TOKEN_LEN} 미만:  {(lengths < MIN_TOKEN_LEN).sum()} / {len(lengths)}")

before = len(ds)
ds = ds.filter(
    lambda ex: MIN_TOKEN_LEN <= ex["n_tokens"] <= MAX_SEQ_LENGTH,
    num_proc=2,
)
after = len(ds)
print(f"\n필터링: {before} → {after} (-{before - after})")

ds = ds.remove_columns(["n_tokens"])

split_ds = ds.train_test_split(test_size=VAL_RATIO, seed=SEED)
train_ds = split_ds["train"]
eval_ds = split_ds["test"]

print(f"\ntrain: {len(train_ds)}")
print(f"eval:  {len(eval_ds)}")
print("\n[train sample preview]")
print(train_ds[0]["text"][:1000])
print("...")

## STEP 8. Gemma 4 chat template 마커 확인

In [ ]:
toy_messages = [
    {"role": "user", "content": "USER_MARKER_12345"},
    {"role": "assistant", "content": "ASSISTANT_MARKER_67890"},
]
toy_text = tokenizer.apply_chat_template(
    toy_messages,
    tokenize=False,
    add_generation_prompt=False,
)

print("=== repr (정확한 문자열) ===")
print(repr(toy_text))
print("\n=== 사람이 읽기 쉽게 ===")
print(toy_text)

INSTRUCTION_PART = "<|turn>user\n"
RESPONSE_PART = "<|turn>model\n"

if INSTRUCTION_PART in toy_text and RESPONSE_PART in toy_text:
    print("\n✅ Gemma 4 마커 확인 통과")
else:
    print("\n⚠️  마커를 못 찾았습니다. 위 repr을 보고 마커를 교체하세요.")
    print(f"    시도: INSTRUCTION_PART = {INSTRUCTION_PART!r}")
    print(f"    시도: RESPONSE_PART = {RESPONSE_PART!r}")

## STEP 9. Trainer 설정 (QLoRA 최적화)

In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback

use_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8

if not use_bf16:
    print("⚠️ bf16 미지원 GPU (T4 등). fp16으로 fallback.")
    print("   fp16은 numerical instability 가능성 있음 → max_grad_norm=1.0이 중요해집니다.")

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_ds,
    eval_dataset = eval_ds,
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LENGTH,
    args = SFTConfig(
        output_dir = OUTPUT_DIR,
        per_device_train_batch_size = PER_DEVICE_BATCH_SIZE,
        per_device_eval_batch_size = PER_DEVICE_BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM,
        warmup_ratio = WARMUP_RATIO,
        num_train_epochs = NUM_EPOCHS,
        learning_rate = LEARNING_RATE,
        bf16 = use_bf16,
        fp16 = not use_bf16,
        max_grad_norm = MAX_GRAD_NORM,
        logging_steps = 10,
        eval_steps = 50,
        save_steps = 50,
        save_strategy = "steps",
        eval_strategy = "steps",
        save_total_limit = 2,
        load_best_model_at_end = True,
        metric_for_best_model = "eval_loss",
        greater_is_better = False,
        optim = OPTIMIZER,              # paged_adamw_8bit (QLoRA 권장)
        weight_decay = WEIGHT_DECAY,
        lr_scheduler_type = "cosine",
        seed = SEED,
        data_seed = SEED,
        report_to = REPORT_TO,
        run_name = RUN_NAME,
        packing = False,
        neftune_noise_alpha = NEFTUNE_ALPHA,
        dataloader_num_workers = 2,
        group_by_length = True,
    ),
    callbacks = [
        EarlyStoppingCallback(
            early_stopping_patience = 3,
            early_stopping_threshold = 0.001,
        ),
    ],
)

try:
    from unsloth.chat_templates import train_on_responses_only

    trainer = train_on_responses_only(
        trainer,
        instruction_part = INSTRUCTION_PART,
        response_part = RESPONSE_PART,
    )
    print("✅ assistant 응답부만 학습 (Gemma 4 template)")
except Exception as e:
    print("⚠️  train_on_responses_only 적용 실패:", e)
    print("    STEP 8 repr 출력을 보고 마커를 재확인하세요.")

### 응답부 학습 검증

In [ ]:
sample = trainer.train_dataset[0]
if "labels" in sample:
    labels = sample["labels"]
    input_ids = sample["input_ids"]
    n_total = len(labels)
    n_masked = sum(1 for l in labels if l == -100)
    n_trained = n_total - n_masked
    print(f"전체 토큰: {n_total}")
    print(f"마스크(−100): {n_masked}  ({100*n_masked/n_total:.1f}%)")
    print(f"학습 대상:    {n_trained}  ({100*n_trained/n_total:.1f}%)")
    print("\n--- 학습 대상 토큰 미리보기 (최초 100개) ---")
    trained_ids = [t for t, l in zip(input_ids, labels) if l != -100][:100]
    print(tokenizer.decode(trained_ids))
    
    if n_trained / n_total < 0.1:
        print("\n⚠️  학습 대상이 10% 미만입니다. 마커가 안 맞을 가능성!")
    elif n_trained / n_total > 0.9:
        print("\n⚠️  거의 모든 토큰이 학습 대상입니다. responses-only가 적용 안 됐을 수 있음.")
else:
    print("⚠️  labels 컬럼이 없습니다.")

## STEP 10. 학습 실행

### 과적합 감지 포인트 (dropout 없이 어떻게?)
- `train_loss` 지속 하락 + `eval_loss` 정체/상승 → 과적합 시작
- EarlyStoppingCallback이 3번 연속 악화 시 자동 중단
- `load_best_model_at_end=True`로 베스트 체크포인트 자동 복원
- NEFTune이 임베딩에 노이즈를 주므로 은근한 정규화 효과

In [ ]:
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    mem_before = torch.cuda.memory_allocated() / 1e9
    print(f"학습 시작 전 GPU 메모리: {mem_before:.2f} GB")

train_result = trainer.train()

print("\n=== 학습 완료 ===")
print(train_result)

print("\n=== 최종 Evaluation ===")
eval_result = trainer.evaluate()
print(eval_result)

## STEP 11. LoRA 저장

In [ ]:
model.save_pretrained("lora_model_gemma4_v4")
tokenizer.save_pretrained("lora_model_gemma4_v4")
print("✅ LoRA 어댑터 저장 완료: ./lora_model_gemma4_v4")

## STEP 12. 추론 테스트

In [ ]:
FastLanguageModel.for_inference(model)

test_prompts = [
    "피보나치 수열의 10번째 항을 단계별로 구해줘.",
    "철수는 사과 3개를 가지고 있었고, 영희에게 사과 1개를 받은 뒤 절반을 먹었습니다. 철수에게 남은 사과는 몇 개인가요? 단계별로 설명해주세요.",
    "이진 탐색이 선형 탐색보다 빠른 이유를 시간 복잡도 관점에서 설명해줘.",
]

USE_THINKING_AT_INFERENCE = False

for i, prompt in enumerate(test_prompts, 1):
    print(f"\n{'='*60}")
    print(f"[테스트 {i}] {prompt}")
    print("="*60)
    
    msgs = []
    if USE_THINKING_AT_INFERENCE:
        msgs.append({"role": "system", "content": "<|think|>"})
    msgs.append({"role": "user", "content": prompt})
    
    inputs = tokenizer.apply_chat_template(
        msgs,
        return_tensors = "pt",
        add_generation_prompt = True,
    ).to("cuda")
    
    out = model.generate(
        input_ids = inputs,
        max_new_tokens = 1024 if USE_THINKING_AT_INFERENCE else 512,
        temperature = 0.3,
        top_p = 0.9,
        do_sample = True,
        repetition_penalty = 1.05,
        pad_token_id = tokenizer.pad_token_id,
    )
    
    generated = out[0][inputs.shape[1]:]
    print(tokenizer.decode(generated, skip_special_tokens=False))

## STEP 13. (선택) HuggingFace 업로드

In [ ]:
# model.push_to_hub("본인HF아이디/ko-reason-gemma4-e4b-qlora-v4", token=HF_TOKEN)
# tokenizer.push_to_hub("본인HF아이디/ko-reason-gemma4-e4b-qlora-v4", token=HF_TOKEN)

# model.push_to_hub_merged(
#     "본인HF아이디/ko-reason-gemma4-e4b-merged-v4",
#     tokenizer,
#     save_method = "merged_16bit",
#     token = HF_TOKEN,
# )

## QLoRA 함정 체크리스트 (이 노트북에서 어떻게 피했는가)

| 함정 | 증상 | 이 노트북의 대응 |
|------|------|------------------|
| LoRA dropout 사용 | 수렴 불안정, 속도 저하 | `lora_dropout=0` |
| 일반 LoRA의 alpha/r 스케일링 불안정 | r 커질수록 학습률 폭주 | `use_rslora=True` |
| 메모리 스파이크 OOM | 긴 시퀀스 만나면 터짐 | `paged_adamw_8bit` |
| fp16 학습 불안정 | NaN loss, gradient 폭주 | bf16 우선 + `max_grad_norm=1.0` |
| full-precision LR 그대로 사용 | 학습 불안정 | LR 낮춤 (3e-5) |
| Prompt까지 학습 | 학습 효율 저하 | `train_on_responses_only` |
| 노이즈 샘플 학습 | 학습 신호 희석 | 길이 필터링 |
| 과적합 감지 실패 | 베스트 지점 놓침 | EarlyStopping + load_best_model |
| packing + responses_only 충돌 | 마스킹 깨짐 | `packing=False` |

## 다음 실험 로드맵

### v4에서 성능 OK → v5로 시도해볼 것
1. **LoRA+ 적용**: `loraplus_lr_ratio=16` (A와 B 매트릭스 학습률 분리)
2. **DoRA**: Unsloth 최신에서 지원. LoRA 대비 +1~2%p 보고
3. **Thinking SFT**: `ENABLE_THINKING_TRAINING=True` + 데이터에 thought 구조화
4. **데이터 확장**: 20k → 50k → 전체
5. **DPO/ORPO 후처리**: SFT 체크포인트에 선호 데이터 얹기

### 평가
- KMMLU, KoBEST 같은 한국어 벤치마크
- Gemma 4는 reasoning 내장 모델이라 단순 chat 점수보다 reasoning 점수 비교 중요

## 참고
- QLoRA 논문: https://arxiv.org/abs/2305.14314 (dropout=0, paged optimizer 권장)
- rsLoRA 논문: https://arxiv.org/abs/2312.03732
- Unsloth Gemma 4 가이드: https://unsloth.ai/docs/models/gemma-4
- NEFTune 논문: https://arxiv.org/abs/2310.05914